In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
import pandas as pd

model_name = "gemma3-27b_finetuned"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,  
    max_seq_length=512,
    load_in_4bit=True        
)


tokenizer = get_chat_template(tokenizer, chat_template="gemma3")


FastLanguageModel.for_inference(model) 


In [ ]:
df = pd.read_csv("2021_Patient_level_prompts.csv")

len(df)

In [ ]:
from tqdm import tqdm  

examples = df['instruction'].tolist()

def format_inference_prompt(example):
    conversation = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": example}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True
    )
    
    return formatted_prompt

formatted_prompts = [format_inference_prompt(example) for example in tqdm(examples)]

tokenized_batch = tokenizer(
    formatted_prompts,
    return_tensors="pt",
    padding=True,              
).to("cuda")

In [ ]:
import re
from tqdm import tqdm

batch_size = 128  
raw_outputs = []  
predictions = []  

tokenizer.padding_side = "left"

answer_pattern = re.compile(r"\b(yes|no)\b", flags=re.IGNORECASE)

def extract_label(text):
    match = answer_pattern.search(text)
    if match:
        return match.group(1).capitalize()
    return text if text else "(empty)"

for i in tqdm(range(0, len(tokenized_batch['input_ids']), batch_size), desc="Generating and Extracting Answers"):

    batch = {key: val[i:i+batch_size] for key, val in tokenized_batch.items()}

    output_batch = model.generate(
        **batch,
        max_length=None,
        max_new_tokens=4,
        do_sample=False,
        use_cache=True,
    )

    prompt_len = batch['input_ids'].shape[1]
    new_tokens = output_batch[:, prompt_len:]

    decoded_batch = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

    for text in decoded_batch:
        text = text.strip()
        raw_outputs.append(text)
        predictions.append(extract_label(text))

df["raw_output"] = raw_outputs
df["prediction"] = predictions

print(df["prediction"].value_counts())

In [ ]:
df.to_csv("Gemma-3-27B_Finetuned_Inference_Results.csv", index=False)